# Week 7 — Multi-Agent Reinforcement Learning
## From Conway to LangGraph: Agent Systems for Physicists in the LLM Era
**Università di Bologna · Dipartimento di Fisica**

---

| Section | Topic | Time |
|---------|-------|------|
| 0 | Installation & imports | 5 min |
| 1 | PettingZoo — the multi-agent API | 10 min |
| 2 | Nash Equilibrium — Q-learners on Rock-Paper-Scissors | 25 min |
| 3 | Grim Trigger — cooperation through credible threats | 20 min |
| 4 | Cooperative Navigation — environment exploration | 15 min |
| 5 | Independent PPO on Cooperative Navigation | 30 min |
| 6 | MADDPG — Centralised Training, Decentralised Execution | 20 min |
| 7 | Bridge to Generative ABM | 10 min |

**Key physics analogies in this notebook:**
- Nash equilibrium ↔ maximum-entropy (Boltzmann) distribution at $T \to \infty$
- Grim Trigger stability ↔ Q-learning discount factor — *identical* recursive inequality
- Non-stationarity ↔ time-varying potential landscape
- MADDPG centralised critic ↔ mean-field theory: global information at train time, local execution at test time
- Cooperative navigation ↔ spontaneous symmetry breaking under shared reward

## Section 0 — Installation & Imports

In [ ]:
# ── install ────────────────────────────────────────────────────────────────────
!pip install pettingzoo[classic,mpe] stable-baselines3 supersuit gymnasium matplotlib numpy --quiet

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ── course plot style ──────────────────────────────────────────────────────────
DARK_BG   = '#1A1A1A'
CARD_BG   = '#2D2D2D'
STEEL     = '#4A9ECC'
GOLD      = '#E8B931'
CORAL     = '#E87040'
GREEN     = '#5CB85C'

plt.rcParams.update({
    'figure.facecolor': DARK_BG,
    'axes.facecolor':   CARD_BG,
    'axes.edgecolor':   '#555555',
    'text.color':       '#DDDDDD',
    'axes.labelcolor':  '#DDDDDD',
    'xtick.color':      '#AAAAAA',
    'ytick.color':      '#AAAAAA',
    'grid.color':       '#3A3A3A',
    'grid.linestyle':   '--',
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'legend.facecolor': '#2D2D2D',
    'legend.edgecolor': '#555555',
})

print('Environment ready.')

---
## Section 1 — PettingZoo: The Multi-Agent API

PettingZoo generalises the Gymnasium API to $N$ agents using the **Agent-Environment Cycle (AEC)** model.
The central difference: instead of one `env.step(action)` per timestep, agents act *sequentially* — each agent
acts and the environment updates before the next agent's turn.

```
Gymnasium (single agent)          PettingZoo AEC (N agents)
─────────────────────────         ──────────────────────────────────
obs  = env.reset()                obs, info = env.reset()
obs, r, done, _, _ =              for agent in env.agent_iter():
    env.step(action)                  obs, r, done, _, _ = env.last()
                                      action = policy[agent](obs)
                                      env.step(action)   # or None if done
```

PettingZoo also provides a **Parallel API** where all agents act simultaneously (we will use it
in Section 5). Key methods are the same; the call signature changes to
`observations, rewards, dones, truncs, infos = env.step(action_dict)`.

**Physics note.** In the Gymnasium world one agent lives in a fixed potential $V(s)$.  
In PettingZoo each agent $i$ lives in a potential that depends on all other agents' policies:
$V_i(s; \pi_{-i})$. When the other agents update their policies, the potential changes — this
is the *non-stationarity* problem that makes convergence so much harder to prove.

In [ ]:
# ── 1.1  AEC loop on Rock-Paper-Scissors ──────────────────────────────────────
from pettingzoo.classic import rps_v2

env = rps_v2.env()
env.reset()

print('Agents        :', env.agents)
print('Observation spaces:')
for ag in env.agents:
    print(f'  {ag}: {env.observation_space(ag)}')
print('Action spaces:')
for ag in env.agents:
    print(f'  {ag}: {env.action_space(ag)}')

# One episode with random play
env.reset()
for agent in env.agent_iter():
    obs, reward, terminated, truncated, info = env.last()
    action = None if (terminated or truncated) else env.action_space(agent).sample()
    env.step(action)

print('\nOne random episode completed.')

In [ ]:
# ── 1.2  Visualise the AEC turn order for 3 rounds ─────────────────────────────
ACTIONS = {0: 'Rock', 1: 'Paper', 2: 'Scissors'}

env = rps_v2.env()
env.reset(seed=42)
history = []

np.random.seed(42)
for agent in env.agent_iter():
    obs, reward, terminated, truncated, info = env.last()
    if terminated or truncated:
        env.step(None)
    else:
        action = env.action_space(agent).sample()
        history.append({'agent': agent, 'action': ACTIONS[action], 'reward': reward})
        env.step(action)

print(f'{'Agent':<12} {'Action':<12} {'Reward':>8}')
print('-' * 34)
for h in history:
    print(f"{h['agent']:<12} {h['action']:<12} {h['reward']:>8.1f}")

print('\nNote: reward=0 until both players have acted (end of round).')

---
## Section 2 — Nash Equilibrium: Q-Learners on Rock-Paper-Scissors

### 2.1  Theory recap

A **Nash Equilibrium** $(\pi_1^*, \pi_2^*, \ldots, \pi_N^*)$ satisfies
$$
V_i(\pi_i^*, \pi_{-i}^*) \;\geq\; V_i(\pi_i, \pi_{-i}^*) \quad \forall\, \pi_i,\; \forall\, i.
$$
It is the multi-agent analogue of the Bellman fixed point $Q^*$:  
just as $Q^*$ is the unique fixed point of $\mathcal{T}$, the Nash equilibrium is the fixed point of the
*joint best-response operator* — no agent can improve by deviating *unilaterally*.

**RPS and maximum entropy.** RPS has no pure-strategy Nash equilibrium.  
The unique Nash equilibrium is the **mixed strategy** $\pi^* = (1/3, 1/3, 1/3)$.  
This is the maximum-entropy distribution over three outcomes:
$$
H(\pi^*) = \log|\mathcal{A}| = \log 3 \approx 1.099 \text{ nats}.
$$
**Physics interpretation.** Any bias (lower entropy) is exploitable.  
The Nash equilibrium is the Boltzmann distribution at $T \to \infty$: all microstates equally probable,
no structure, no exploitable pattern. Rational play under perfect mutual information forces maximum disorder.

### 2.2  Two independent Q-learners — will they find Nash?

In [ ]:
# ── 2.2  Two independent Q-learners on RPS ────────────────────────────────────
#
# RPS is a one-state, 3-action game.  Q-table = vector of length 3 per agent.
# We use softmax (Boltzmann) exploration with temperature tau annealed to 0.

N_EPISODES   = 60_000
ALPHA        = 0.05      # learning rate
GAMMA        = 0.0       # one-shot game: no future
TAU_START    = 2.0
TAU_END      = 0.05

# Payoff matrix for agent 0 (agent 1 gets negative)
#   rows = agent0 action (0=R,1=P,2=S), cols = agent1 action
PAYOFF = np.array([
    [ 0, -1,  1],   # Rock    vs R/P/S
    [ 1,  0, -1],   # Paper
    [-1,  1,  0],   # Scissors
], dtype=float)

def softmax(q, tau):
    q = q - q.max()
    e = np.exp(q / tau)
    return e / e.sum()

def entropy(pi):
    pi = np.clip(pi, 1e-10, 1)
    return -np.sum(pi * np.log(pi))

rng = np.random.default_rng(0)
Q   = [np.zeros(3), np.zeros(3)]  # Q[agent][action]

log_every  = 500
entropy_history = [[], []]
q_history       = [[], []]

for ep in range(1, N_EPISODES + 1):
    tau = TAU_START * (TAU_END / TAU_START) ** (ep / N_EPISODES)

    pis  = [softmax(Q[i], tau) for i in range(2)]
    acts = [rng.choice(3, p=pis[i]) for i in range(2)]

    r0  = PAYOFF[acts[0], acts[1]]
    r1  = -r0

    Q[0][acts[0]] += ALPHA * (r0 - Q[0][acts[0]])
    Q[1][acts[1]] += ALPHA * (r1 - Q[1][acts[1]])

    if ep % log_every == 0:
        for i in range(2):
            pi = softmax(Q[i], TAU_END)  # evaluation policy
            entropy_history[i].append(entropy(pi))
            q_history[i].append(Q[i].copy())

steps = np.arange(1, len(entropy_history[0]) + 1) * log_every
NASH_H = np.log(3)

print(f'Nash entropy target : {NASH_H:.4f} nats')
for i in range(2):
    pi_final = softmax(Q[i], TAU_END)
    print(f'Agent {i} final policy: R={pi_final[0]:.3f}  P={pi_final[1]:.3f}  S={pi_final[2]:.3f}  '
          f'H={entropy(pi_final):.4f}')

In [ ]:
# ── 2.3  Plot: entropy convergence + Q-value oscillation ─────────────────────
fig = plt.figure(figsize=(14, 5), facecolor=DARK_BG)
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# Left panel: entropy vs episodes
ax0 = fig.add_subplot(gs[0])
colors = [STEEL, GOLD]
for i in range(2):
    ax0.plot(steps, entropy_history[i], color=colors[i], lw=2, label=f'Agent {i}')
ax0.axhline(NASH_H, color=CORAL, ls='--', lw=1.5, label=f'Nash  H=log(3)={NASH_H:.3f}')
ax0.set_xlabel('Episode')
ax0.set_ylabel('Policy entropy H(π)  [nats]')
ax0.set_title('Convergence to Nash (maximum entropy)')
ax0.legend()
ax0.grid(True)
ax0.set_ylim(0, 1.25)

# Right panel: Q-value trajectories for agent 0
ax1 = fig.add_subplot(gs[1])
q_arr = np.array(q_history[0])  # shape (T, 3)
action_names = ['Rock', 'Paper', 'Scissors']
q_colors = [STEEL, GOLD, GREEN]
for a in range(3):
    ax1.plot(steps, q_arr[:, a], color=q_colors[a], lw=1.5, label=action_names[a])
ax1.axhline(0, color='#666666', ls='--', lw=1)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Q-value')
ax1.set_title('Agent 0 Q-values — oscillation = non-stationarity')
ax1.legend()
ax1.grid(True)

fig.suptitle('Nash Equilibrium via Independent Q-Learning on RPS',
             color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('rps_nash.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nKey observation: entropy → log(3) but Q-values never stop oscillating.')
print('The policy converges; the Q-values do not. This is the hallmark of MARL non-stationarity.')

---
## Section 3 — Grim Trigger: Cooperation Through Credible Threats

### 3.1  Theory: iterated Prisoner's Dilemma

The **Prisoner's Dilemma** payoff matrix (row = agent 0, col = agent 1):

|  | Cooperate | Defect |
|--|-----------|--------|
| **Cooperate** | $R, R$ = (3, 3) | $S, T$ = (0, 5) |
| **Defect** | $T, S$ = (5, 0) | $P, P$ = (1, 1) |

One-shot Nash: mutual defection $(P, P) = (1, 1)$ — suboptimal for both.  
Pareto optimum: mutual cooperation $(R, R) = (3, 3)$.

**Grim Trigger** is the simplest strategy that sustains cooperation in the *repeated* game:
> Cooperate until the opponent defects. If the opponent ever defects, defect forever.

**When is Grim Trigger a Nash equilibrium?** A defecting deviation gives immediate gain
$T - R$ but triggers permanent punishment. The deviation is not profitable iff:
$$
T - R \;\leq\; \frac{\gamma}{1 - \gamma}(R - P)
\quad\Longleftrightarrow\quad
\gamma \;\geq\; \frac{T - R}{T - P}.
$$

**This is identical to the Q-learning discount factor condition** (same recursive inequality structure).  
With $T=5, R=3, P=1$: critical $\gamma^* = (5-3)/(5-1) = 0.5$.  
For $\gamma \geq 0.5$, cooperation is the Nash equilibrium of the infinite repeated game.

**Physics analogy.** Cooperation is the *ordered phase*; defection is the *disordered phase*.
The discount factor $\gamma$ plays the role of *inverse temperature* $\beta = 1/T$.
At $\gamma < \gamma^*$ (high temperature) the system is disordered; at $\gamma > \gamma^*$ it orders.
The Grim Trigger condition is the *phase boundary* — a cooperation–defection phase transition.

In [ ]:
# ── 3.2  Grim Trigger simulation ──────────────────────────────────────────────
#
# We simulate 4 strategy pairs over T rounds:
#   - Grim Trigger vs Grim Trigger
#   - Tit-for-Tat vs Tit-for-Tat
#   - Always Defect vs Always Defect
#   - Grim Trigger vs Always Defect

T_ROUNDS  = 50
N_TRIALS  = 500

T_reward, R_reward, P_reward, S_reward = 5, 3, 1, 0

PAYOFF_PD = np.array([
    [R_reward, S_reward],  # row agent cooperates: opp C or D
    [T_reward, P_reward],  # row agent defects
])

def simulate_ipd(strategy0, strategy1, rounds, seed=None):
    rng = np.random.default_rng(seed)
    rewards0, rewards1 = [], []
    history0, history1 = [], []
    triggered0 = triggered1 = False

    for t in range(rounds):
        # --- agent 0 decision ---
        if strategy0 == 'grim':
            a0 = 1 if triggered0 else 0  # 0=C, 1=D
        elif strategy0 == 'tft':
            a0 = history1[-1] if history1 else 0
        elif strategy0 == 'allD':
            a0 = 1
        elif strategy0 == 'allC':
            a0 = 0
        else:
            a0 = rng.integers(0, 2)

        # --- agent 1 decision ---
        if strategy1 == 'grim':
            a1 = 1 if triggered1 else 0
        elif strategy1 == 'tft':
            a1 = history0[-1] if history0 else 0
        elif strategy1 == 'allD':
            a1 = 1
        elif strategy1 == 'allC':
            a1 = 0
        else:
            a1 = rng.integers(0, 2)

        r0 = PAYOFF_PD[a0, a1]
        r1 = PAYOFF_PD[a1, a0]

        rewards0.append(r0); rewards1.append(r1)
        history0.append(a0); history1.append(a1)

        if a1 == 1: triggered0 = True
        if a0 == 1: triggered1 = True

    return np.array(rewards0), np.array(rewards1)

scenarios = [
    ('Grim vs Grim',     'grim', 'grim'),
    ('TfT  vs TfT',      'tft',  'tft'),
    ('AllD vs AllD',     'allD', 'allD'),
    ('Grim vs AllD',     'grim', 'allD'),
]

results = {}
for label, s0, s1 in scenarios:
    r0s, r1s = zip(*[simulate_ipd(s0, s1, T_ROUNDS, seed=k) for k in range(N_TRIALS)])
    results[label] = {
        'mean_r0': np.mean([r.mean() for r in r0s]),
        'mean_r1': np.mean([r.mean() for r in r1s]),
        'cumulative_curve': np.mean([np.cumsum(r) for r in r0s], axis=0)
    }
    print(f'{label:<20}  avg_reward_agent0={results[label]["mean_r0"]:.3f}  '
          f'avg_reward_agent1={results[label]["mean_r1"]:.3f}')

In [ ]:
# ── 3.3  Plot cumulative reward curves ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK_BG)

# Panel A: cumulative reward per scenario
ax = axes[0]
palette = [STEEL, GREEN, CORAL, GOLD]
for (label, s0, s1), color in zip(scenarios, palette):
    ax.plot(results[label]['cumulative_curve'], color=color, lw=2, label=label)

# Reference: Pareto optimum (both always cooperate)
ax.plot(np.arange(1, T_ROUNDS+1) * R_reward, color='white', lw=1, ls=':', label=f'Pareto (R={R_reward}/round)')
ax.plot(np.arange(1, T_ROUNDS+1) * P_reward, color='#555555', lw=1, ls=':', label=f'Nash defection (P={P_reward}/round)')
ax.set_xlabel('Round')
ax.set_ylabel('Cumulative reward (agent 0)')
ax.set_title('Iterated Prisoner\'s Dilemma — Cumulative Reward')
ax.legend(fontsize=9)
ax.grid(True)

# Panel B: phase diagram — cooperation as function of gamma
ax2 = axes[1]
gammas = np.linspace(0, 1, 500)
# Sustained cooperation condition: gamma >= (T-R)/(T-P)
gamma_star = (T_reward - R_reward) / (T_reward - P_reward)
coop_region = gammas >= gamma_star

ax2.fill_between(gammas, 0, 1, where=~coop_region, alpha=0.25, color=CORAL, label='Defection dominates')
ax2.fill_between(gammas, 0, 1, where=coop_region,  alpha=0.25, color=GREEN, label='Cooperation sustained')
ax2.axvline(gamma_star, color=GOLD, lw=2, ls='--', label=f'γ* = {gamma_star:.2f}')

# Grim Trigger stability boundary: (T-R)/(T-P)
ax2.annotate(f'Phase transition\nγ* = (T−R)/(T−P)\n    = {gamma_star:.2f}',
             xy=(gamma_star, 0.5), xytext=(gamma_star + 0.08, 0.5),
             color=GOLD, fontsize=10,
             arrowprops=dict(arrowstyle='->', color=GOLD))

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_xlabel('Discount factor γ')
ax2.set_ylabel('')
ax2.set_yticks([])
ax2.set_title('Grim Trigger Phase Diagram  (T=5, R=3, P=1)')
ax2.legend(loc='lower right')
ax2.grid(True, axis='x')

fig.suptitle('Cooperation as an Ordered Phase: Grim Trigger Analysis',
             color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('grim_trigger.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Critical γ* = {gamma_star:.3f}  (same formula as Q-learning discount threshold)')

---
## Section 4 — Cooperative Navigation: Environment Exploration

**Cooperative Navigation** (`simple_spread_v3` in PettingZoo MPE) is a benchmark where
$N$ agents must cover $N$ landmarks while avoiding collisions.

- **State**: each agent observes its own velocity, its displacement to all landmarks,
  and the relative positions of other agents  
- **Action**: 5 discrete actions — no-op, left, right, up, down
- **Reward**: shared global reward = $-$(sum of minimum agent–landmark distances) $-$ collision penalty  
- **Challenge**: the reward is *fully shared* — a gift to free-riders, a challenge for coordination

**Physics framing.** The shared reward creates a joint free energy
$\mathcal{F}(\pi_1, \ldots, \pi_N)$. Optimising it is analogous to finding the ground
state of a coupled system: the landmarks act as potential-energy minima, and agents
must spontaneously break the permutation symmetry (each agent selects *one* landmark to cover).
Without coordination, the system gets stuck in a symmetric saddle point — all agents
crowding the same landmark.

In [ ]:
# ── 4.1  Inspect simple_spread_v3 ─────────────────────────────────────────────
from pettingzoo.mpe import simple_spread_v3

N_AGENTS = 3
env = simple_spread_v3.env(N=N_AGENTS, local_ratio=0.5, max_cycles=25, continuous_actions=False)
env.reset(seed=42)

print('Agents :', env.agents)
for ag in env.agents:
    obs_sp = env.observation_space(ag)
    act_sp = env.action_space(ag)
    print(f'  {ag}: obs_shape={obs_sp.shape}  action_space={act_sp}')

# Observation breakdown for 3 agents, 3 landmarks:
# 2 (own vel) + 2 (own pos) + 3*2 (landmark displacements) + 2*(N-1)*2 (other agents) = 18
obs_size = env.observation_space('agent_0').shape[0]
print(f'\nObservation vector ({obs_size} dims):')
print('  [0:2]   own velocity (vx, vy)')
print('  [2:4]   own position (x, y)  — not in partial obs variants')
print('  [4:10]  displacements to 3 landmarks (dx, dy) × 3')
print('  [10:18] relative positions of other 2 agents (dx, dy) × 2')

In [ ]:
# ── 4.2  Random baseline — measure how bad uncoordinated agents are ─────────────
from pettingzoo.mpe import simple_spread_v3

N_EVAL = 200
env = simple_spread_v3.parallel_env(N=3, local_ratio=0.5, max_cycles=25, continuous_actions=False)

ep_returns = []
for ep in range(N_EVAL):
    obs, infos = env.reset(seed=ep)
    total_reward = 0.0
    while env.agents:
        actions = {ag: env.action_space(ag).sample() for ag in env.agents}
        obs, rewards, terms, truncs, infos = env.step(actions)
        total_reward += sum(rewards.values()) / len(rewards)
    ep_returns.append(total_reward)

print(f'Random policy — mean episode return: {np.mean(ep_returns):.2f} ± {np.std(ep_returns):.2f}')
print(f'(Lower is better — reward is negative distance; target is 0)')

In [ ]:
# ── 4.3  Visualise a random episode trajectory ────────────────────────────────
env = simple_spread_v3.parallel_env(N=3, local_ratio=0.5, max_cycles=25, continuous_actions=False)
obs, infos = env.reset(seed=7)

reward_history = []
while env.agents:
    actions = {ag: env.action_space(ag).sample() for ag in env.agents}
    obs, rewards, terms, truncs, infos = env.step(actions)
    reward_history.append(list(rewards.values()))

r_arr = np.array(reward_history)   # shape (T, N_agents)

fig, ax = plt.subplots(figsize=(10, 4), facecolor=DARK_BG)
colors = [STEEL, GOLD, GREEN]
for i in range(r_arr.shape[1]):
    ax.plot(r_arr[:, i], color=colors[i], alpha=0.7, lw=1.5, label=f'agent_{i}')
ax.plot(r_arr.mean(axis=1), color='white', lw=2.5, label='mean', zorder=5)
ax.axhline(0, color='#666666', ls='--')
ax.set_xlabel('Step')
ax.set_ylabel('Reward (negative distance)')
ax.set_title('Random episode — Cooperative Navigation (3 agents, 3 landmarks)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('coop_nav_random.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 5 — Independent PPO on Cooperative Navigation

**Independent PPO (IPPO)** is the simplest multi-agent extension: treat each agent as
an independent single-agent PPO learner, ignoring the fact that the other agents exist.

| Property | IPPO |
|----------|------|
| Policy | Each agent has its own MLP |
| Critic | Each agent's critic sees only its own observation |
| Assumption | Stationarity (violated in MARL) |
| Convergence | No guarantee — but often works in practice |
| Scalability | $O(N)$ parameters, $O(1)$ inference |

**Why does it work at all?** Despite the non-stationarity, if agents share the
reward signal and their policies are initialised symmetrically, the gradient signal
points in the same direction for all agents. The system can coordinate through
*emergent specialisation*: different random initialisations lead different agents
to cover different landmarks — spontaneous symmetry breaking under shared reward.

**Implementation.** We use [SuperSuit](https://github.com/Farama-Foundation/SuperSuit) to
convert the PettingZoo Parallel environment into a vectorised SB3-compatible environment.
All agents share the **same network weights** (parameter sharing = implicit communication).

In [ ]:
# ── 5.1  Wrap environment for SB3 with SuperSuit ──────────────────────────────
import supersuit as ss
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

def make_coop_env(n_agents=3, n_cpus=1, n_envs=4):
    env = simple_spread_v3.parallel_env(
        N=n_agents, local_ratio=0.5, max_cycles=25, continuous_actions=False
    )
    # pad observations so all agents have the same shape (required for parameter sharing)
    env = ss.pad_observations_v0(env)
    # treat each agent as a separate 'process' in the vectorised env
    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(env, n_envs, num_cpus=n_cpus, base_class='stable_baselines3')
    return env

env = make_coop_env(n_agents=3, n_cpus=1, n_envs=4)
print('Vectorised env — obs shape :', env.observation_space.shape)
print('               — act space :', env.action_space)

In [ ]:
# ── 5.2  Reward-logging callback ──────────────────────────────────────────────
class RewardLogger(BaseCallback):
    def __init__(self, log_every=5000):
        super().__init__()
        self.log_every = log_every
        self.ep_rewards = []
        self._buf = []

    def _on_step(self):
        for info in self.locals['infos']:
            if 'episode' in info:
                self._buf.append(info['episode']['r'])
        if self.n_calls % self.log_every == 0 and self._buf:
            self.ep_rewards.append(np.mean(self._buf))
            self._buf = []
        return True

In [ ]:
# ── 5.3  Train IPPO ───────────────────────────────────────────────────────────
#
# Training budget: 300k timesteps (~5 min on Colab CPU).
# Each timestep = one agent's action; 3 agents × 25 steps = 75 agent-steps per episode.

TOTAL_STEPS = 300_000

callback = RewardLogger(log_every=5000)

model = PPO(
    'MlpPolicy', env,
    learning_rate=3e-4,
    n_steps=128,
    batch_size=256,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.01,
    verbose=0,
)

print(f'Training IPPO for {TOTAL_STEPS:,} steps ...')
model.learn(total_timesteps=TOTAL_STEPS, callback=callback)
print('Training complete.')

# Save model
model.save('ippo_coop_nav')
print('Model saved to ippo_coop_nav.zip')

In [ ]:
# ── 5.4  Evaluate trained IPPO vs random baseline ─────────────────────────────
from pettingzoo.mpe import simple_spread_v3
from stable_baselines3 import PPO

model = PPO.load('ippo_coop_nav')

def evaluate_policy(policy_fn, n_episodes=200, seed_offset=0):
    env_eval = simple_spread_v3.parallel_env(N=3, local_ratio=0.5,
                                             max_cycles=25, continuous_actions=False)
    returns = []
    for ep in range(n_episodes):
        obs_dict, _ = env_eval.reset(seed=ep + seed_offset)
        total = 0.0
        while env_eval.agents:
            actions = {}
            for ag in env_eval.agents:
                a, _ = policy_fn(obs_dict[ag])
                actions[ag] = a
            obs_dict, rewards, terms, truncs, _ = env_eval.step(actions)
            total += sum(rewards.values()) / 3
        returns.append(total)
    return np.array(returns)

def random_policy(obs):
    return np.random.randint(0, 5), None

def ippo_policy(obs):
    action, _ = model.predict(obs, deterministic=True)
    return int(action), None

ret_random = evaluate_policy(random_policy, n_episodes=200)
ret_ippo   = evaluate_policy(ippo_policy,   n_episodes=200)

print(f'Random policy : {ret_random.mean():.2f} ± {ret_random.std():.2f}')
print(f'IPPO          : {ret_ippo.mean():.2f}   ± {ret_ippo.std():.2f}')
print(f'Improvement   : {ret_ippo.mean() - ret_random.mean():+.2f}')

In [ ]:
# ── 5.5  Plot training curve + return distributions ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK_BG)

# Panel A: training curve
ax = axes[0]
if callback.ep_rewards:
    steps_arr = np.arange(1, len(callback.ep_rewards) + 1) * 5000
    ax.plot(steps_arr, callback.ep_rewards, color=STEEL, lw=2)
    # smoothed
    w = max(1, len(callback.ep_rewards)//10)
    smooth = np.convolve(callback.ep_rewards, np.ones(w)/w, mode='valid')
    ax.plot(steps_arr[w-1:], smooth, color=GOLD, lw=2.5, label='smoothed')
ax.set_xlabel('Timestep')
ax.set_ylabel('Mean episode reward')
ax.set_title('IPPO Training Curve — Cooperative Navigation')
ax.grid(True)
ax.legend()

# Panel B: return distributions
ax2 = axes[1]
bins = np.linspace(min(ret_random.min(), ret_ippo.min()),
                   max(ret_random.max(), ret_ippo.max()), 40)
ax2.hist(ret_random, bins=bins, color=CORAL, alpha=0.6, label='Random')
ax2.hist(ret_ippo,   bins=bins, color=STEEL, alpha=0.6, label='IPPO')
ax2.axvline(ret_random.mean(), color=CORAL, lw=2, ls='--')
ax2.axvline(ret_ippo.mean(),   color=STEEL, lw=2, ls='--')
ax2.set_xlabel('Episode return')
ax2.set_ylabel('Count')
ax2.set_title('Return Distributions: Random vs IPPO')
ax2.legend()
ax2.grid(True)

fig.suptitle('Independent PPO — Cooperative Navigation (3 agents)',
             color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('ippo_results.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 6 — MADDPG: Centralised Training, Decentralised Execution

### 6.1  The CTDE paradigm

IPPO's critical flaw: each agent's critic $V_i(o_i)$ sees only its *own* observation,
making the environment non-stationary from each agent's perspective.

**MADDPG** (Lowe et al., 2017) breaks the symmetry between training and execution:

| Phase | Information available | What uses it |
|-------|-----------------------|-------------|
| **Training** | Global state + all agents' actions | Centralised critic $Q_i(\mathbf{o}, \mathbf{a})$ |
| **Execution** | Agent $i$'s local observation only | Decentralised actor $\mu_i(o_i)$ |

The centralised critic $Q_i(o_1, \ldots, o_N, a_1, \ldots, a_N)$ is **stationary**:
it observes the full joint state, so the environment looks stationary to it.
At execution time, only the actor $\mu_i(o_i)$ is deployed — no centralised information needed.

**Physics analogy: mean-field theory.**  
The centralised critic is exactly the mean-field approximation:  
at train time, every agent "sees" the average field generated by all other agents (the joint critic).  
At test time, each agent runs locally under the frozen field.  
CTDE is mean-field theory operationalised as a training algorithm.

### 6.2  MADDPG architecture

```
TRAINING (centralised)                EXECUTION (decentralised)
──────────────────────────────────    ─────────────────────────────
                                      
  o1 ─► actor_1 ─► a1 ──────────────► a1 ─► environment
  o2 ─► actor_2 ─► a2               
  o3 ─► actor_3 ─► a3               
         │          │
  [o1,o2,o3, a1,a2,a3]
         │
  critic_1(o1..3, a1..3) ──► Q1
  critic_2(o1..3, a1..3) ──► Q2     (Q2, Q3 not needed at exec time)
  critic_3(o1..3, a1..3) ──► Q3
```

**Policy gradient for agent $i$:**
$$
\nabla_{\theta_i} J = \mathbb{E}\bigl[
  \nabla_{\theta_i} \log \pi_{\theta_i}(a_i | o_i)
  \cdot Q_i^\pi(\mathbf{o}, \mathbf{a})
\bigr]
$$
The critic $Q_i^\pi$ uses the joint observation-action pair $\mathbf{o}, \mathbf{a}$
— it provides a *stationary* gradient signal even though the individual agents are moving.

In [ ]:
# ── 6.3  MADDPG — conceptual implementation sketch ───────────────────────────
#
# Full MADDPG training requires a custom replay buffer and N×2 networks.
# Here we implement the *data flow* to make the architecture concrete,
# without running a full training loop.
import torch
import torch.nn as nn

N_AGENTS   = 3
OBS_DIM    = 18   # per agent (simple_spread_v3, N=3)
ACT_DIM    = 5    # discrete actions → one-hot for critic input

class Actor(nn.Module):
    """Decentralised actor: maps agent i's local observation to action logits."""
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, act_dim),
        )
    def forward(self, obs):
        return self.net(obs)   # logits — apply softmax for probabilities

class CentralisedCritic(nn.Module):
    """Centralised critic: maps (ALL obs, ALL actions) → Q-value for agent i."""
    def __init__(self, n_agents, obs_dim, act_dim, hidden=64):
        super().__init__()
        # input: concat of all observations + all action one-hots
        in_dim = n_agents * (obs_dim + act_dim)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, all_obs, all_acts):
        # all_obs:  (batch, N, obs_dim)
        # all_acts: (batch, N, act_dim)
        x = torch.cat([all_obs, all_acts], dim=-1)  # (batch, N, obs+act)
        x = x.view(x.shape[0], -1)                  # (batch, N*(obs+act))
        return self.net(x)                           # (batch, 1)

# Instantiate one actor and critic per agent
actors  = [Actor(OBS_DIM, ACT_DIM) for _ in range(N_AGENTS)]
critics = [CentralisedCritic(N_AGENTS, OBS_DIM, ACT_DIM) for _ in range(N_AGENTS)]

# Count parameters
total_actor  = sum(p.numel() for a in actors  for p in a.parameters())
total_critic = sum(p.numel() for c in critics for p in c.parameters())

print(f'Architecture summary  (N_agents={N_AGENTS})')
print(f'  Actor  (per agent) : {sum(p.numel() for p in actors[0].parameters()):,} params')
print(f'  Critic (per agent) : {sum(p.numel() for p in critics[0].parameters()):,} params')
print(f'  Total actors       : {total_actor:,} params')
print(f'  Total critics      : {total_critic:,} params')
print(f'  Grand total        : {total_actor + total_critic:,} params')
print()
print('Critic input dim:', N_AGENTS * (OBS_DIM + ACT_DIM),
      f'= {N_AGENTS} agents × ({OBS_DIM} obs + {ACT_DIM} act)')

In [ ]:
# ── 6.4  Forward pass demonstration ──────────────────────────────────────────
BATCH = 32

# Simulate a batch from replay buffer
obs_batch  = torch.randn(BATCH, N_AGENTS, OBS_DIM)   # all agents' observations
# Each actor produces logits; softmax to get pseudo-one-hots for critic
act_logits = [actors[i](obs_batch[:, i, :]) for i in range(N_AGENTS)]
act_probs  = [torch.softmax(logits, dim=-1) for logits in act_logits]
acts_batch = torch.stack(act_probs, dim=1)           # (batch, N, act_dim)

# Each centralised critic evaluates Q for the FULL joint state
for i, critic in enumerate(critics):
    q_val = critic(obs_batch, acts_batch)
    print(f'  Critic {i} — Q-value batch shape: {q_val.shape}  '
          f'mean={q_val.mean().item():.4f}')

print()
print('Note: the critic input is ALL observations + ALL actions.')
print('This is what makes the environment look stationary to the critic.')

In [ ]:
# ── 6.5  IPPO vs MADDPG comparison summary ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4), facecolor=DARK_BG)
ax.axis('off')

headers = ['Property', 'IPPO (Independent PPO)', 'MADDPG (CTDE)']
rows = [
    ['Critic input',          'Local obs  o_i',         'Joint obs+actions  (o, a)'],
    ['Actor input',           'Local obs  o_i',         'Local obs  o_i'],
    ['Non-stationarity',      'Visible (problem)',       'Hidden by joint critic'],
    ['Training complexity',   'O(N) independent runs',  'O(N²) joint critic input'],
    ['Physics analogy',       'N decoupled particles',  'Mean-field theory'],
    ['Convergence guarantee', 'None in general',        'None, but more stable'],
    ['Sample efficiency',     'Moderate',               'High (off-policy replay)'],
    ['Scalability to N>>3',   'Good',                   'Poor (critic grows as N²)'],
]

col_widths = [0.22, 0.35, 0.35]
col_x      = [0.01, 0.24, 0.61]

y0 = 0.92
dy = 0.10

# Header
for j, (h, x, w) in enumerate(zip(headers, col_x, col_widths)):
    color = STEEL if j == 1 else (GOLD if j == 2 else '#FFFFFF')
    ax.text(x, y0, h, transform=ax.transAxes,
            fontsize=10, fontweight='bold', color=color, va='top')

ax.axhline(y0 - 0.02, xmin=0, xmax=1, color='#555555', transform=ax.transAxes, lw=1)

for k, row in enumerate(rows):
    y = y0 - (k + 1) * dy
    bg_color = CARD_BG if k % 2 == 0 else DARK_BG
    ax.add_patch(plt.Rectangle((0, y - 0.04), 1, dy, transform=ax.transAxes,
                                color=bg_color, zorder=0))
    for j, (cell, x) in enumerate(zip(row, col_x)):
        color = '#DDDDDD' if j == 0 else (STEEL if j == 1 else GOLD)
        ax.text(x, y, cell, transform=ax.transAxes,
                fontsize=9, color=color, va='top')

ax.set_title('IPPO vs MADDPG — Architecture Comparison',
             color='white', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig('ippo_vs_maddpg.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 7 — Bridge to Generative Agent-Based Models

### 7.1  The limits of policy-gradient agents

IPPO and MADDPG agents are powerful — but brittle in a specific way:
their behaviour is fully determined by a fixed neural network trained on a
specific reward signal in a specific environment.  
They cannot:
- **Reason** in natural language
- **Plan** over long horizons without a reward signal
- **Communicate** rich intentional content
- **Generalise** to novel tasks without retraining

### 7.2  The generative ABM paradigm

In a **Generative ABM** (Park et al., 2023; Argyle et al., 2023), agents are
not policies — they are *language model instances* with memory and goals.

```
Traditional MARL agent             Generative ABM agent
─────────────────────────          ────────────────────────────────
State:     obs vector              State:  memory stream (text)
Policy:    MLP(obs) → action       Policy: LLM(context, memory) → text → action
Reward:    scalar signal           Reward: none (or implicit via reflection)
Learning:  gradient descent        Learning: in-context, no weight update
```

The key question GABM asks: **do LLM agents reproduce emergent social phenomena
when placed in a structured environment?**
- Do they segregate (like Schelling agents) without being told to?
- Do they cooperate (Grim Trigger-like) without an explicit discount factor?
- Do they form opinion clusters (Axelrod cultural diffusion)?

### 7.3  Synthetic populations

A **synthetic population** is a set of LLM agents each conditioned on
a demographic-statistical profile drawn from survey data.  
The profile specifies age, political affiliation, income, education, etc.  
The LLM then *role-plays* the corresponding individual — not as a character,
but as a statistical representative of a real population segment.

Application: predicting survey responses, policy reactions, electoral outcomes
by *simulating* the population rather than polling it.

**Physics framing.**  
A synthetic population is a microstate ensemble:  
each agent is a degree of freedom drawn from the empirical distribution $P(\text{demographics})$.  
The macroscopic observable (e.g., policy approval) is the ensemble average — the thermal average
in the corresponding statistical-mechanics language.

In [ ]:
# ── 7.4  Mini synthetic-population demo ──────────────────────────────────────
#
# A tiny synthetic population (no LLM required).
# We draw agent profiles from a hypothetical demographic distribution
# and show how ensemble averages recover the target statistics.

rng = np.random.default_rng(42)
N_POP = 1000

# Demographic distribution (loosely inspired by EU survey data)
ages         = rng.integers(18, 80, N_POP)
incomes      = rng.lognormal(mean=3.5, sigma=0.8, size=N_POP)   # k€/year
edu_levels   = rng.choice([0, 1, 2], N_POP, p=[0.30, 0.45, 0.25])  # 0=HS 1=BSc 2=MSc+
pol_affinity = rng.normal(0, 1, N_POP)    # -2=left ... +2=right

# Synthetic survey: "Do you support Policy X (renewable energy subsidy)?" (0/1)
# Older, higher income, right-leaning → less support
logit = (0.5 - 0.003 * ages - 0.01 * incomes + 0.5 * edu_levels
         - 0.4 * pol_affinity + rng.normal(0, 0.5, N_POP))
support = (logit > 0).astype(int)

print(f'Synthetic population  N={N_POP}')
print(f'  Overall support: {support.mean()*100:.1f}%')
print(f'  Under-30 support: {support[ages<30].mean()*100:.1f}%')
print(f'  Over-60 support:  {support[ages>60].mean()*100:.1f}%')
print(f'  High-edu support: {support[edu_levels==2].mean()*100:.1f}%')
print(f'  Low-edu support:  {support[edu_levels==0].mean()*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor=DARK_BG)

# Age distribution coloured by support
for val, color, label in [(1, STEEL, 'Support'), (0, CORAL, 'Oppose')]:
    axes[0].hist(ages[support==val], bins=20, color=color, alpha=0.65, label=label, density=True)
axes[0].set_title('Age distribution by support')
axes[0].set_xlabel('Age')
axes[0].legend()
axes[0].grid(True)

# Political affinity
for val, color, label in [(1, STEEL, 'Support'), (0, CORAL, 'Oppose')]:
    axes[1].hist(pol_affinity[support==val], bins=25, color=color, alpha=0.65,
                 label=label, density=True)
axes[1].set_title('Political affinity by support')
axes[1].set_xlabel('Political leaning (left ← → right)')
axes[1].legend()
axes[1].grid(True)

# Support rate by education
edu_labels = ['HS', 'BSc', 'MSc+']
edu_support = [support[edu_levels==k].mean()*100 for k in range(3)]
bars = axes[2].bar(edu_labels, edu_support, color=[STEEL, GOLD, GREEN], width=0.5)
axes[2].set_title('Support rate by education level')
axes[2].set_ylabel('Support (%)')
axes[2].set_ylim(0, 100)
axes[2].axhline(support.mean()*100, color=CORAL, ls='--', label='Overall mean')
axes[2].legend()
axes[2].grid(True, axis='y')

fig.suptitle('Synthetic Population — Demographic Heterogeneity',
             color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('synthetic_pop.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.5  Course arc summary ───────────────────────────────────────────────────
arc = [
    ('Week 1–2', 'Cellular Automata',    'Local rules → global patterns'),
    ('Week 3–4', 'Agent-Based Models',   'Mesa, Schelling, Vicsek, sandpile'),
    ('Week 5',   'Tabular RL',           'MDP, Q-learning, SARSA'),
    ('Week 6',   'Deep RL',              'DQN, PPO, SAC (GT Sophy)'),
    ('Week 7',   'Multi-Agent RL',       'Nash, Grim Trigger, IPPO, MADDPG'),
    ('Week 8+',  'Generative ABM',       'LLM agents, synthetic populations'),
]

print(f'{"Phase":<10}  {"Topic":<22}  Highlights')
print('─' * 70)
for phase, topic, highlights in arc:
    print(f'{phase:<10}  {topic:<22}  {highlights}')

print()
print('The thread: from deterministic local rules (CA) → stochastic agents (ABM)')
print('→ optimising agents (RL) → rational strategic agents (MARL)')
print('→ language-capable social agents (GABM).')
print()
print('Each step adds one degree of freedom to the agent model.')
print('GABM is the limit where the agent model is complex enough to simulate a human.')

---
## Exercises

**Exercise 1 — Nash entropy (25 min)**  
Modify the RPS Q-learner to use $\varepsilon$-greedy instead of softmax exploration.
Does the policy still converge to the Nash entropy $\log 3$?  
Plot the entropy trajectory for both exploration schemes.
*Concept tested:* exploration strategy, Nash convergence, entropy.

**Exercise 2 — Phase transition (30 min)**  
Verify the Grim Trigger phase boundary numerically:
simulate the iterated PD for $\gamma \in \{0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9\}$
where both agents play Grim Trigger but with probability $p=0.05$ of making a mistake
(accidental defection). Plot mean cooperation rate vs $\gamma$. Where is the transition?  
*Concept tested:* discount factor, phase transition, noise robustness.

**Exercise 3 ★ — Critic input ablation (45 min)**  
Implement a toy centralised critic for 2 agents in a 1D environment.  
Compare two variants: (A) critic sees joint observation $(o_1, o_2)$ but not actions;
(B) critic sees $(o_1, o_2, a_1, a_2)$.  
Which gives lower variance gradient estimates? Measure empirically.  
*Concept tested:* MADDPG design, variance reduction, gradient estimation.

**Exercise 4 ★★ — GABM opinion dynamics (60 min)**  
Use the synthetic population framework from Section 7.4.  
Implement a simple Deffuant opinion dynamics model where two agents interact
if their political affinity difference is $< \epsilon$, and update their beliefs toward each other.  
Vary $\epsilon \in [0.3, 0.5, 0.7, 1.0]$ and plot the opinion cluster structure.
Compare with the Ising/Potts mean-field prediction.  
*Concept tested:* opinion dynamics, bounded confidence model, phase transitions in social systems.

---
*From Conway to LangGraph: Agent Systems for Physicists in the LLM Era*  
*Università di Bologna · Dipartimento di Fisica · Week 7*